<a href="https://colab.research.google.com/github/lab-rasool/SIIM/blob/main/notebooks/SIIM_LocalLLMs_Backup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIIM 2026 Learning Lab — Backup Cloud Instance
### Running Local LLMs Behind Institutional Firewalls (LL4022)

This notebook is a **fallback** for the hands-on labs. If a laptop won't
cooperate during the workshop, run these cells top to bottom to get:

- **Ollama** serving an open model (Lab 1)
- **OpenWebUI** — a private, ChatGPT-style interface, reachable at a public URL (Lab 2)
- the **Lab 3 clinical workflows** (radiology summarization + pathology extraction)

---

> ## ⚠️ Read this first — this is the *opposite* of "behind the firewall"
> A Colab instance is a **public cloud machine**. It is the right tool for a
> conference demo with **synthetic data**, and the wrong tool for anything real.
>
> **Use synthetic data only. Never paste real patient data or PHI into this
> notebook or the public OpenWebUI URL it creates.** The whole point of the
> Learning Lab is that the production pattern keeps the model *inside* your
> network — this backup deliberately steps outside it, so treat everything
> here as public.
>
> The public URL is **unauthenticated** — the workshop build skips the
> OpenWebUI login so there is nothing to type, which also means anyone who has
> the link can use the chat. The URL is random and one-off. Shut the runtime
> down (Runtime → Disconnect and delete runtime) when you're done so the
> tunnel closes.

## Step 0 — Pick your model(s)
`llama3.2` matches Lab 1 (~2 GB, fast). Set `LARGE_MODEL` to also pull a
bigger model for the side-by-side comparison in Lab 3 — leave it blank to skip.

**Tip:** use a GPU runtime (Runtime → Change runtime type → T4 GPU) so the
models run quickly. Ollama uses the GPU automatically when one is present.

In [1]:
# Workshop model configuration
SMALL_MODEL = "llama3.2"        # Lab 1 default (~2 GB)
LARGE_MODEL = "qwen2.5:7b"     # for the small-vs-large comparison; set to "" to skip

import os
os.environ["SMALL_MODEL"] = SMALL_MODEL
os.environ["LARGE_MODEL"] = LARGE_MODEL

# Ollama performance tuning. Every `ollama serve` we start later inherits this
# environment, so set it once, up front:
#   FLASH_ATTENTION  -> faster, lower-memory attention kernels
#   KV_CACHE_TYPE    -> q8_0 quantizes the KV cache (needs flash attention),
#                       ~halving its memory so longer reports fit on a T4
#   KEEP_ALIVE=-1    -> keep the model resident in VRAM between requests, so
#                       there is no reload pause during the live demo
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"]   = "q8_0"
os.environ["OLLAMA_KEEP_ALIVE"]      = "-1"

print("Will pull:", SMALL_MODEL, "and", LARGE_MODEL or "(no large model)")


Will pull: llama3.2 and qwen2.5:7b


## Step 1 — Install Ollama and pull the model(s)
Installs the Ollama runtime, starts it in the background, and pulls your
model(s). On a T4 GPU this takes a couple of minutes.

**Use a GPU runtime before running this:** Runtime → Change runtime type →
**T4 GPU**. Ollama uses the GPU automatically *once it can detect it* — the
cell below installs the detection tools (`pciutils`, `lshw`) so it can. On a
CPU runtime `llama3.2` still works but is slow, and a 7B model is impractical
for a live demo (set `LARGE_MODEL = ""` in Step 0).

In [2]:
# Ollama's installer ships zstd-compressed archives and detects GPUs via
# lspci/lshw - a fresh Colab runtime has none of these, so install them first.
# Without zstd the install fails; without pciutils/lshw it silently falls back
# to CPU even on a GPU runtime.
!sudo apt-get update -qq
!sudo apt-get install -y -qq zstd pciutils lshw

# Install the Ollama runtime
!curl -fsSL https://ollama.com/install.sh | sh

# Make sure the binary actually landed before we try to run it
import shutil, subprocess, time, os
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
assert os.path.exists(ollama_bin), (
    'Ollama did not install. Re-run this cell; if it persists, check that '
    'the zstd install above succeeded.'
)

# Is a GPU actually attached? If this errors, you're on a CPU runtime:
# Runtime -> Change runtime type -> T4 GPU, then re-run from this cell.
# (llama3.2 still works on CPU; a 7B model will be very slow.)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU detected - running on CPU.'

# Colab has no systemd, so start the server ourselves in the background.
# This subprocess inherits the OLLAMA_* perf env set in Step 0.
subprocess.Popen([ollama_bin, 'serve'])
time.sleep(8)

# Pull the workshop model(s)
!ollama pull $SMALL_MODEL
if os.environ.get("LARGE_MODEL"):
    get_ipython().system('ollama pull $LARGE_MODEL')

# Confirm what's running locally (Lab 1, step 3)
!curl -s localhost:11434/api/tags | python3 -m json.tool


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 6.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package pci.ids.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../0-pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../1-libpci3_1%3a3.7.0-6_amd64.deb ...
Unp

## Step 2 - Install OpenWebUI
OpenWebUI needs Python 3.11. We use [uv](https://docs.astral.sh/uv/) to do this
in one fast step: uv fetches a managed CPython 3.11 itself (no `apt`) and builds
an isolated virtual environment at `/content/venv`, then installs OpenWebUI into
it much faster than pip. This cell only *installs* - we start the servers a
couple of cells down.

In [3]:
# Install OpenWebUI with uv. uv creates the Python 3.11 venv at an ABSOLUTE
# path AND fetches a managed CPython 3.11 itself, so the old apt python3.11
# steps are gone. Later cells still expect /content/venv, so the path is
# unchanged.
!pip install -q uv

# Absolute path /content/venv (NOT a relative "venv") so it's found no matter
# what the working directory is when later cells run. --python 3.11 makes uv
# download a managed 3.11 build if the runtime lacks one.
!export UV_VENV_CLEAR=1
!uv venv --python 3.11 /content/venv
!uv pip install --python /content/venv/bin/python open-webui -q
print("OpenWebUI installed at /content/venv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 80.3 MB/s eta 0:00:00
Using CPython 3.11.15
Creating virtual environment at: venv
Activate with: source venv/bin/activate
OpenWebUI installed at /content/venv


## Step 3 — Clone the workshop repo
Brings in the Lab 3 scripts and the **synthetic** clinical datasets.

In [4]:
import os, glob

# Always use an absolute repo path and only clone if it isn't already there.
# (Re-running this cell will NOT create nested SIIM/SIIM folders.)
REPO_DIR = "/content/SIIM"
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !rm -rf {REPO_DIR}
    !git clone https://github.com/lab-rasool/SIIM.git {REPO_DIR}

# Find where the lab scripts actually live — repo root or a subfolder —
# so this works however the code was pushed to the repo.
hits = glob.glob(os.path.join(REPO_DIR, '**', 'summarize_report.py'), recursive=True)
LAB_DIR = os.path.dirname(hits[0]) if hits else REPO_DIR
os.environ['LAB_DIR'] = LAB_DIR
%cd {LAB_DIR}
print('Lab scripts dir:', LAB_DIR)
!ls -1

Cloning into '/content/SIIM'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 87 (delta 32), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 53.24 KiB | 3.33 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/SIIM
Lab scripts dir: /content/SIIM
data
extract_pathology.py
notebooks
README.md
requirements.txt
summarize_report.py
utils


## Step 4 — Start OpenWebUI and get a public link
Click this one cell. It starts Ollama, pulls the model, starts the private
ChatGPT-style interface **connected to Ollama**, and prints a public link like
`https://....trycloudflare.com` via a [Cloudflare](https://www.cloudflare.com/)
quick tunnel (no install, no account). **If no link appears, just click the cell
again** — there is nothing to type.

Open the link and you go **straight into the chat** — the OpenWebUI login is
turned off for the workshop, so there is no sign-up step. Your pulled model is
pre-selected in the dropdown at the top-left. OpenWebUI reaches Ollama
*server-side* over localhost, so the model works the same through the public
link as it does on the box.

> The link is **public and unauthenticated** — anyone who has it can use this
> chat. The URL is random and disappears when you stop the runtime. Use
> **synthetic data only**, and shut the runtime down when you are done
> (Runtime → Disconnect and delete runtime).

In [ ]:
# Makes OpenWebUI work end-to-end for the workshop: starts Ollama, pulls the
# model, starts OpenWebUI (login disabled, so there's nothing to type), and
# opens a public link via Pinggy's Python SDK. Click to run. If no link appears,
# just run it again. No typing required, start to finish.
import os, re, sys, time, shutil, subprocess, urllib.request

def _up(url):
    try: urllib.request.urlopen(url, timeout=3); return True
    except Exception: return False

# 1) Make sure Ollama is running, then pull the model so OpenWebUI has something.
#    (serve inherits the OLLAMA_* perf env set in Step 0.)
ob = shutil.which("ollama") or "/usr/local/bin/ollama"
if not _up("http://localhost:11434/api/tags") and os.path.exists(ob):
    subprocess.Popen([ob, "serve"])
    for _ in range(20):
        if _up("http://localhost:11434/api/tags"): break
        time.sleep(1)
MODEL = os.environ.get("SMALL_MODEL", "llama3.2")
if _up("http://localhost:11434/api/tags"):
    subprocess.run(["ollama", "pull", MODEL])

# 2) Start OpenWebUI, pointed at the local Ollama. For a hands-on workshop we
#    set WEBUI_AUTH=False so there's NO sign-up step - the link opens straight
#    into the chat. DEFAULT_MODELS pre-selects your pulled model so the dropdown
#    is ready. (Auth-off works because each Colab runtime is a fresh install.)
#    OpenWebUI talks to Ollama server-side over localhost, so it works the same
#    whether you reach the UI locally or through the public link.
if not os.path.exists("/content/venv/bin/open-webui"):
    print("\u26a0\ufe0f OpenWebUI isn't installed yet - run Step 2 first, then this cell.")
else:
    if not _up("http://localhost:8081"):
        env = dict(os.environ,
                   OLLAMA_BASE_URL="http://127.0.0.1:11434",
                   ENABLE_OLLAMA_API="true",
                   WEBUI_AUTH="False",
                   DEFAULT_MODELS=MODEL)
        subprocess.Popen(["/content/venv/bin/open-webui", "serve", "--port", "8081"],
                         env=env, stdout=open("/content/openwebui.log","w"),
                         stderr=subprocess.STDOUT)
        print("Starting OpenWebUI\u2026 first boot can take ~30-60s.")
        for _ in range(60):
            if _up("http://localhost:8081"): break
            time.sleep(2)

    # 3) Public link via Pinggy's Python SDK. Unlike the raw `ssh free@a.pinggy.io`
    #    command, the SDK manages the tunnel connection itself (with retries) and
    #    returns the URL directly via tunnel.urls - no log scraping, and it isn't
    #    at the mercy of Colab's ssh client stalling on one edge node. Only the
    #    UI (8081) is forwarded; Ollama (11434) stays private. Free tier: random
    #    URL, 60-min sessions, no account.
    try:
        import pinggy
    except ImportError:
        print("Installing the Pinggy SDK (one-time)\u2026")
        subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q", "pinggy"])
        import pinggy

    # Close any tunnel left over from a previous run of this cell.
    try:
        _prev = globals().get("_pinggy_tunnel")
        if _prev is not None: _prev.stop()
    except Exception:
        pass

    url = None
    for attempt in range(1, 4):                      # up to 3 tries
        try:
            _pinggy_tunnel = pinggy.start_tunnel(forwardto="localhost:8081")
            for _ in range(15):                      # wait for the URL to populate
                urls = list(getattr(_pinggy_tunnel, "urls", []) or [])
                https = [u for u in urls if u.startswith("https://")]
                if https: url = https[0]; break
                if urls:  url = urls[0];  break
                time.sleep(2)
            if url: break
        except Exception as e:
            print(f"   Pinggy attempt {attempt} didn't connect ({e}) - retrying\u2026")
            time.sleep(3)

    if url and _up("http://localhost:8081"):
        print("\n\u2705 OpenWebUI is ready - open this link:\n   " + url)
        print("   (Pinggy shows a one-time 'Enter Site' splash - click through it.)")
        print("   It then opens straight into the chat (no login). Your model is")
        print("   pre-selected at the top-left; refresh once if it's not there yet.")
    elif url:
        print("\nLink is up but OpenWebUI is still booting - open it in ~30s:\n   " + url)
    else:
        print("\nNo public link yet - just run this cell again.")
        print("(If it keeps failing, the assigned Pinggy edge may be busy: Runtime \u2192")
        print(" Disconnect and delete runtime, reconnect, and run the notebook again.)")

Starting OpenWebUI… first boot can take ~30-60s.


## Step 5 — Run the Lab 3 clinical workflows
Each cell below makes sure Ollama is running and the model is downloaded
before it runs, so you can just click them — in any order, even if you skipped
a step. All data is synthetic; nothing real goes in.

In [6]:
# Radiology summarization — makes sure Ollama + the model are ready first.
import os, time, glob, shutil, subprocess, urllib.request

MODEL = os.environ.get("SMALL_MODEL", "llama3.2")
LAB   = os.environ.get("LAB_DIR", "")
if not LAB or not os.path.exists(os.path.join(LAB, "summarize_report.py")):
    hits = glob.glob("/content/SIIM/**/summarize_report.py", recursive=True)
    LAB = os.path.dirname(hits[0]) if hits else "/content/SIIM"

def _up():
    try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3); return True
    except Exception: return False

if not _up():
    ob = shutil.which("ollama") or "/usr/local/bin/ollama"
    if os.path.exists(ob):
        subprocess.Popen([ob, "serve"])
        for _ in range(20):
            if _up(): break
            time.sleep(1)

if _up():
    subprocess.run(["ollama", "pull", MODEL])   # instant if already downloaded
    get_ipython().system(f'python3 "{LAB}/summarize_report.py" --model {MODEL} '
                         f'--report "{LAB}/data/radiology/ct_chest_001.txt"')
else:
    print("⚠️ Ollama isn't ready. Please run Step 1 (Install Ollama), then this cell.")


Loaded 1 synthetic radiology report(s). Host: http://localhost:11434

REPORT: ct_chest_001.txt

INPUT REPORT
------------------------------------------------------------------------------
SYNTHETIC RADIOLOGY REPORT — FOR EDUCATIONAL USE ONLY (NO PHI)
Accession: SYN-RAD-0001  |  Patient: SYNTHETIC, PATIENT A  |  MRN: 000-00-0001

EXAM: CT CHEST WITH CONTRAST

CLINICAL HISTORY: 64-year-old with incidental pulmonary nodule on prior imaging.
Follow-up to assess for interval change.

TECHNIQUE: Helical CT of the chest was performed following administration of
intravenous contrast. Axial images were reconstructed at 1.25 mm and 5 mm.

FINDINGS:
Lungs: There is a 9 mm spiculated nodule in the right upper lobe, previously
measuring 6 mm, demonstrating interval growth. No additional suspicious
pulmonary nodules. No pleural effusion. No consolidation.
Mediastinum: No mediastinal or hilar lymphadenopathy. Heart size normal.
No pericardial effusion.
Bones: Mild degenerative changes of the thoraci

In [7]:
# Pathology extraction — makes sure Ollama + the model are ready first.
import os, time, glob, shutil, subprocess, urllib.request

MODEL = os.environ.get("SMALL_MODEL", "llama3.2")
LAB   = os.environ.get("LAB_DIR", "")
if not LAB or not os.path.exists(os.path.join(LAB, "extract_pathology.py")):
    hits = glob.glob("/content/SIIM/**/extract_pathology.py", recursive=True)
    LAB = os.path.dirname(hits[0]) if hits else "/content/SIIM"

def _up():
    try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3); return True
    except Exception: return False

if not _up():
    ob = shutil.which("ollama") or "/usr/local/bin/ollama"
    if os.path.exists(ob):
        subprocess.Popen([ob, "serve"])
        for _ in range(20):
            if _up(): break
            time.sleep(1)

if _up():
    subprocess.run(["ollama", "pull", MODEL])
    get_ipython().system(f'python3 "{LAB}/extract_pathology.py" --model {MODEL} '
                         f'--report "{LAB}/data/pathology/path_lung_002.txt"')
else:
    print("⚠️ Ollama isn't ready. Please run Step 1 (Install Ollama), then this cell.")


Loaded 1 synthetic pathology report(s). Model: llama3.2. Host: http://localhost:11434

REPORT: path_lung_002.txt

INPUT REPORT
------------------------------------------------------------------------------
SYNTHETIC PATHOLOGY REPORT — FOR EDUCATIONAL USE ONLY (NO PHI)
Accession: SYN-PATH-0002  |  Patient: SYNTHETIC, PATIENT F  |  MRN: 000-00-0012

SPECIMEN: Right upper lobe, wedge resection.

CLINICAL HISTORY: Enlarging right upper lobe nodule.

FINAL DIAGNOSIS:
Lung, right upper lobe, wedge resection:
- ADENOCARCINOMA, acinar predominant, moderately differentiated.
- Tumor size: 2.2 cm in greatest dimension.
- Visceral pleura: not involved.
- Margins: negative; closest margin 0.8 cm.
- Lymphovascular invasion: not identified.
- Pathologic stage (this specimen): pT1c. Lymph nodes not submitted.

COMMENT: Molecular studies for EGFR, ALK, and PD-L1 are pending and will be
reported in an addendum.

EXTRACTED FIELDS (llama3.2)
--------------------------------------------------------------

In [8]:
# Small vs. larger model, side by side. Needs LARGE_MODEL set in Step 0.
import os, time, glob, shutil, subprocess, urllib.request

MODEL = os.environ.get("SMALL_MODEL", "llama3.2")
LAB   = os.environ.get("LAB_DIR", "")
if not LAB or not os.path.exists(os.path.join(LAB, "summarize_report.py")):
    hits = glob.glob("/content/SIIM/**/summarize_report.py", recursive=True)
    LAB = os.path.dirname(hits[0]) if hits else "/content/SIIM"

def _up():
    try: urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3); return True
    except Exception: return False

if not _up():
    ob = shutil.which("ollama") or "/usr/local/bin/ollama"
    if os.path.exists(ob):
        subprocess.Popen([ob, "serve"])
        for _ in range(20):
            if _up(): break
            time.sleep(1)

LARGE = os.environ.get("LARGE_MODEL", "")
if not LARGE:
    print("Set LARGE_MODEL in Step 0 (and run it) to enable the comparison.")
elif _up():
    subprocess.run(["ollama", "pull", MODEL])
    subprocess.run(["ollama", "pull", LARGE])
    get_ipython().system(f'python3 "{LAB}/summarize_report.py" --compare {MODEL} {LARGE} '
                         f'--report "{LAB}/data/radiology/mri_brain_003.txt"')
else:
    print("⚠️ Ollama isn't ready. Please run Step 1 (Install Ollama), then this cell.")


Loaded 1 synthetic radiology report(s). Host: http://localhost:11434

REPORT: mri_brain_003.txt

INPUT REPORT
------------------------------------------------------------------------------
SYNTHETIC RADIOLOGY REPORT — FOR EDUCATIONAL USE ONLY (NO PHI)
Accession: SYN-RAD-0003  |  Patient: SYNTHETIC, PATIENT C  |  MRN: 000-00-0003

EXAM: MRI BRAIN WITH AND WITHOUT CONTRAST

CLINICAL HISTORY: 47-year-old with new-onset headaches and transient left-sided
weakness.

TECHNIQUE: Multiplanar, multisequence MRI of the brain before and after
gadolinium administration.

FINDINGS:
There is a 2.1 cm rim-enhancing mass in the right frontal lobe with surrounding
vasogenic edema and mild local mass effect on the adjacent sulci. No midline
shift. No hydrocephalus. No restricted diffusion to suggest acute infarct.
No abnormal susceptibility to suggest hemorrhage. The ventricles are normal in
size. Major intracranial flow voids are preserved.

IMPRESSION:
1. 2.1 cm rim-enhancing right frontal mass with 